In [3]:
from ultralytics import YOLO
import cv2
import numpy as np


model = YOLO("/mnt/EBI-SHARE/06_Data/labelstudio/models/multiclass/2025_07_21_01_32_56_model/weights/best.pt")
input_path = "/mnt/EBI-SHARE/06_Data/labelstudio/datasets/evaluation/leuthener_benchmark.mp4"
output_path = "/mnt/EBI-SHARE/06_Data/labelstudio/models/multiclass/2025_07_21_01_32_56_model/leuthener_benchmark_predictions.mp4"

# Video öffnen mit OpenCV, um Metadaten auszulesen
cap = cv2.VideoCapture(input_path)
fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
cap.release()

# VideoWriter initialisieren
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

# Inferenz mit YOLOv8s-seg Modell auf Video-Datei
results = model.predict(
    source=input_path,
    stream=True,
    conf=0.7  # nur Vorhersagen mit Confidence > 0.5
)

for r in results:
    # Annotiertes Frame holen (mit Segmentations-Masken falls vorhanden)
    frame = r.plot()
    frame_bgr = cv2.cvtColor(frame, cv2.COLOR_RGB2BGR)  # RGB → BGR für OpenCV
    out.write(frame_bgr)

# Writer schließen
out.release()
print(f"✅ Annotated video saved to {output_path}")


video 1/1 (frame 1/1960) /mnt/EBI-SHARE/06_Data/labelstudio/datasets/evaluation/leuthener_benchmark.mp4: 416x640 (no detections), 31.1ms
video 1/1 (frame 2/1960) /mnt/EBI-SHARE/06_Data/labelstudio/datasets/evaluation/leuthener_benchmark.mp4: 416x640 (no detections), 19.9ms
video 1/1 (frame 3/1960) /mnt/EBI-SHARE/06_Data/labelstudio/datasets/evaluation/leuthener_benchmark.mp4: 416x640 (no detections), 19.8ms
video 1/1 (frame 4/1960) /mnt/EBI-SHARE/06_Data/labelstudio/datasets/evaluation/leuthener_benchmark.mp4: 416x640 (no detections), 19.8ms
video 1/1 (frame 5/1960) /mnt/EBI-SHARE/06_Data/labelstudio/datasets/evaluation/leuthener_benchmark.mp4: 416x640 (no detections), 20.0ms
video 1/1 (frame 6/1960) /mnt/EBI-SHARE/06_Data/labelstudio/datasets/evaluation/leuthener_benchmark.mp4: 416x640 (no detections), 19.9ms
video 1/1 (frame 7/1960) /mnt/EBI-SHARE/06_Data/labelstudio/datasets/evaluation/leuthener_benchmark.mp4: 416x640 (no detections), 19.9ms
video 1/1 (frame 8/1960) /mnt/EBI-SHARE/

In [2]:
from ultralytics import YOLO
import cv2

# Load YOLO models
model1 = YOLO("/mnt/EBI-SHARE/06_Data/labelstudio/models/doors/best.pt")
model2 = YOLO("/mnt/EBI-SHARE/06_Data/labelstudio/models/windows/best.pt")

# Open video
cap = cv2.VideoCapture("/mnt/EBI-SHARE/06_Data/labelstudio/datasets/evaluation/leuthener_benchmark.mp4")
fps = cap.get(cv2.CAP_PROP_FPS)
w, h = int(cap.get(3)), int(cap.get(4))
out = cv2.VideoWriter("output.mp4", cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))

# Process video frame by frame
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    # Inference on both models
    results1 = model1(frame, verbose=False)[0]
    results2 = model2(frame, verbose=False)[0]

    # Draw results from both models (different colors)
    frame = results1.plot(line_width=2, boxes={"color": (255, 255, 0)}, labels=True)
    frame = results2.plot(line_width=1, boxes={"color": (255, 0, 0)}, labels=True)  # second model in blue

    out.write(frame)

# Cleanup
cap.release()
out.release()

In [ ]:
import supervision as sv
import numpy as np
from ultralytics import YOLO

VIDEO_PATH = "/mnt/EBI-SHARE/06_Data/labelstudio/datasets/evaluation/leuthener_benchmark.mp4"

model = YOLO("yolov8s.pt")

video_info = sv.VideoInfo.from_video_path(VIDEO_PATH)

def process_frame(frame: np.ndarray, _) -> np.ndarray:
    results = model(frame, imgsz=1280)[0]
    
    detections = sv.Detections.from_yolov8(results)

    box_annotator = sv.BoxAnnotator(thickness=4, text_thickness=4, text_scale=2)

    labels = [f"{model.names[class_id]} {confidence:0.2f}" for _, _, confidence, class_id, _ in detections]
    frame = box_annotator.annotate(scene=frame, detections=detections, labels=labels)

    return frame

sv.process_video(source_path=VIDEO_PATH, target_path=f"result.mp4", callback=process_frame)


100%|██████████| 21.5M/21.5M [00:00<00:00, 75.2MB/s]


Exception: Could not open video at video.mp4

In [ ]:
import supervision as sv
import numpy as np
from ultralytics import YOLO

VIDEO_PATH = "/mnt/EBI-SHARE/06_Data/labelstudio/datasets/evaluation/leuthener_benchmark.mp4"

# List your model paths here
model_paths = [
    "/mnt/EBI-SHARE/06_Data/labelstudio/models/doors/best.pt",
    "/mnt/EBI-SHARE/06_Data/labelstudio/models/windows/best.pt",
]

models = [YOLO(path) for path in model_paths]
video_info = sv.VideoInfo.from_video_path(VIDEO_PATH)

def process_frame(frame: np.ndarray, _) -> np.ndarray:
    for model in models:
        results = model(frame, imgsz=1920)[0]
        detections = sv.Detections.from_ultralytics(results)
        # Filter out detections with None class_id
        valid = [i for i, cid in enumerate(detections.class_id) if cid is not None]
        if len(valid) == 0:
            continue  # Skip if no valid detections
        detections.xyxy = detections.xyxy[valid]
        detections.class_id = detections.class_id[valid]
        detections.confidence = detections.confidence[valid]
        box_annotator = sv.BoxAnnotator(thickness=4)
        labels = [
            f"{model.names[class_id]} {confidence:.2f}"
            for class_id, confidence in zip(detections.class_id, detections.confidence)
        ]
        frame = box_annotator.annotate(frame, detections, labels)
    return frame

sv.process_video(source_path=VIDEO_PATH, target_path="result.mp4", callback=process_frame)


0: 1216x1920 2 Doors, 22.0ms
Speed: 4.7ms preprocess, 22.0ms inference, 0.9ms postprocess per image at shape (1, 3, 1216, 1920)


TypeError: '<' not supported between instances of 'NoneType' and 'int'